# Personal Expense Data Analyzer

Python-based expense analysis project for VS Code/Jupyter.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Personal Expense Data Analyzer
# Run: python expense_analyzer.py

os.makedirs("charts", exist_ok=True)
os.makedirs("reports", exist_ok=True)

df = pd.read_csv("expenses.csv")

print("\n===== PERSONAL EXPENSE DATA ANALYZER =====")
print(f"Rows loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")

# Data preprocessing
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

print("\nMissing values before cleaning:")
print(df.isnull().sum())

duplicate_count = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicate_count}")

df = df.dropna(subset=["Date", "Amount"]).copy()
df["Category"] = df["Category"].astype(str).str.strip().str.title()
df["Payment_Method"] = df["Payment_Method"].astype(str).str.strip().str.upper()
df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Year"] = df["Date"].dt.year

# Basic statistics
total = df["Amount"].sum()
average = df["Amount"].mean()
median = df["Amount"].median()
mode = df["Amount"].mode().iloc[0] if not df["Amount"].mode().empty else np.nan
minimum = df["Amount"].min()
maximum = df["Amount"].max()
variance = df["Amount"].var()
std_dev = df["Amount"].std()
data_range = maximum - minimum
q1 = df["Amount"].quantile(0.25)
q3 = df["Amount"].quantile(0.75)
iqr = q3 - q1

print("\n===== DESCRIPTIVE STATISTICS =====")
print(f"Total Expense       : ₹{total:.2f}")
print(f"Average Expense     : ₹{average:.2f}")
print(f"Median Expense      : ₹{median:.2f}")
print(f"Mode Expense        : ₹{mode:.2f}")
print(f"Minimum Expense     : ₹{minimum:.2f}")
print(f"Maximum Expense     : ₹{maximum:.2f}")
print(f"Variance            : {variance:.2f}")
print(f"Standard Deviation  : {std_dev:.2f}")
print(f"Range               : ₹{data_range:.2f}")
print(f"IQR                 : ₹{iqr:.2f}")

# Grouped analysis
category_total = df.groupby("Category")["Amount"].sum().sort_values(ascending=False)
category_count = df.groupby("Category").size().sort_values(ascending=False)
monthly_total = df.groupby("Month")["Amount"].sum()
payment_total = df.groupby("Payment_Method")["Amount"].sum().sort_values(ascending=False)
payment_count = df.groupby("Payment_Method").size().sort_values(ascending=False)

print("\n===== CATEGORY-WISE TOTAL =====")
print(category_total)

print("\n===== MONTHLY TOTAL =====")
print(monthly_total)

print("\n===== PAYMENT METHOD TOTAL =====")
print(payment_total)

# Save reports
summary = pd.DataFrame({
    "Metric": [
        "Valid Transactions", "Total Expense", "Average Expense",
        "Median Expense", "Minimum Expense", "Maximum Expense",
        "Variance", "Standard Deviation", "Range", "IQR"
    ],
    "Value": [
        len(df), total, average, median, minimum, maximum,
        variance, std_dev, data_range, iqr
    ]
})
summary.to_csv("reports/expense_summary.csv", index=False)

category_report = pd.DataFrame({
    "Category": category_total.index,
    "Total_Amount": category_total.values,
    "Transaction_Count": [category_count.get(x, 0) for x in category_total.index]
})
category_report["Percentage_of_Total"] = (
    category_report["Total_Amount"] / total * 100
)
category_report.to_csv("reports/category_summary.csv", index=False)

monthly_total.reset_index().rename(columns={"Amount":"Total_Amount"}).to_csv(
    "reports/monthly_summary.csv", index=False
)

payment_report = pd.DataFrame({
    "Payment_Method": payment_total.index,
    "Total_Amount": payment_total.values,
    "Transaction_Count": [payment_count.get(x, 0) for x in payment_total.index]
})
payment_report.to_csv("reports/payment_method_summary.csv", index=False)

# Charts
plt.figure(figsize=(8, 5))
category_total.plot(kind="bar")
plt.title("Category-wise Expense")
plt.xlabel("Category")
plt.ylabel("Amount (₹)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("charts/category_wise_expense.png", dpi=150)
plt.close()

plt.figure(figsize=(8, 5))
monthly_total.plot(kind="line", marker="o")
plt.title("Monthly Expense Trend")
plt.xlabel("Month")
plt.ylabel("Amount (₹)")
plt.grid(True)
plt.tight_layout()
plt.savefig("charts/monthly_expense.png", dpi=150)
plt.close()

plt.figure(figsize=(7, 5))
payment_total.plot(kind="bar")
plt.title("Expense by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Amount (₹)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("charts/payment_method.png", dpi=150)
plt.close()

print("\nCharts saved in charts/")
print("Reports saved in reports/")
print("Analysis completed successfully.")
